# Data Cleaning & Processing

## Imports

In [18]:
from pathlib import Path

import pandas as pd
from IPython.display import display

## Load the Data

In [19]:
cwd = Path.cwd()
root = next(p for p in [cwd, *cwd.parents] if (p / "data" / "raw" / "survey_data.csv").exists())

survey_path = root / "data" / "raw" / "survey_data.csv"
codelist_path = root / "data" / "raw" / "codelist.csv"
df = pd.read_csv(survey_path)
codelist = pd.read_csv(codelist_path)

display(df.head(5))
display(codelist)

,Respondent,Sex,Location,Q1,Q2,Q3,Q4,Q5
0,1,M,MT,4,3,1,1,5
1,2,F,CO,2,1,1,2,2
2,3,M,CO,3,3,2,3,4
3,4,M,MT,3,1,1,3,2
4,5,M,MT,2,2,1,4,1


,Unnamed: 0,Question,1,2,3,4,5
0,Q1,How familiar are you with the Inflation Reduct...,Very Aware,Somewhat Aware,Not really aware,Not aware at all,Don't know
1,Q2,How familiar are you with Green Machine?,Very Aware,Somewhat Aware,Not really aware,Not aware at all,Don't know
2,Q3,How familiar are you with the World Wildlife F...,Very Aware,Somewhat Aware,Not really aware,Not aware at all,Don't know
3,Q4,What is the most important enviornmental issue...,Climate Change,Environmental Justice,Clean Air & Water,Land & Wildlife,Something Else / Don't Know
4,Q5,How likely are you to donate to an environment...,Extremely Likely,Very Likely,Somewhat Likely,Not likely,Unsure


## Standardize Column Names

In [39]:
# Using a copy of the original for editing/cleaning
df_clean = df.copy()
# Clean up any column names with spaces or any trailing spaces
df_clean.columns = (df_clean.columns.str.strip().str.lower().str.replace(r"[^\w]+", "_", regex=True).str.strip("_"))
# Display new column names
print("Standardized columns:", df_clean.columns.tolist())

Standardized columns: ['respondent', 'sex', 'location', 'q1', 'q2', 'q3', 'q4', 'q5']


## Missingness and Duplicates

Logging any rows with missing information, but given this is a test dataset, it looks pristine! (Was not expecting that)

In [26]:
missing_summary = pd.DataFrame({"missing_count": df_clean.isna().sum(), "missing_percentage": df_clean.isna().mean().mul(100).round(2), "dtype": df_clean.dtypes.astype(str)}).sort_values("missing_count", ascending=False)
display(missing_summary)
print("Duplicate rows:", df_clean.duplicated().sum())

,missing_count,missing_percentage,dtype
respondent,0,0.0,int64
sex,0,0.0,object
location,0,0.0,object
q1,0,0.0,int64
q2,0,0.0,int64
q3,0,0.0,int64
q4,0,0.0,int64
q5,0,0.0,int64


Duplicate rows: 0


## Recode Q4 as a Categorical Label

In the original dataframe, Q4 (Most important environment issue) was coded numerically, giving the false assumption it should be treated as an `ordinal` type. The code below changes the datatype from `int` to `object`. 

In [40]:
q4_row = codelist.loc[codelist.iloc[:, 0].astype(str).str.strip().str.upper() == "Q4"]
q4_row = q4_row.iloc[0]
q4_map = {int(code): str(q4_row[str(code)]).strip() for code in range(1, 6)}
df_clean["q4"] = pd.to_numeric(df_clean["q4"], errors="raise").map(q4_map)
display(df_clean["q4"].value_counts(dropna=False).rename("count").to_frame())

,count
q4,
Land & Wildlife,29
Climate Change,28
Clean Air & Water,24
Environmental Justice,16
Something Else / Don't Know,3


## Final Structure and Saved Output

Displaying the final cleaed data output. It doesn't differ too much from the original data, besides standardized column names and a new Q4 datatype

In [41]:
display(df_clean.head())
print("Final shape:", df_clean.shape)
print("Final dtypes:")
print(df_clean.dtypes)
clean_path = root / "data" / "clean" / "clean_data.csv"
clean_path.parent.mkdir(parents=True, exist_ok=True)
df_clean.to_csv(clean_path, index=False)
print(f"Saved cleaned data to {clean_path}")

,respondent,sex,location,q1,q2,q3,q4,q5
0,1,M,MT,4,3,1,Climate Change,5
1,2,F,CO,2,1,1,Environmental Justice,2
2,3,M,CO,3,3,2,Clean Air & Water,4
3,4,M,MT,3,1,1,Clean Air & Water,2
4,5,M,MT,2,2,1,Land & Wildlife,1


Final shape: (100, 8)
Final dtypes:
respondent     int64
sex           object
location      object
q1             int64
q2             int64
q3             int64
q4            object
q5             int64
dtype: object
Saved cleaned data to /Users/iansargent/Precision_Interview_Assessment/data/clean/clean_data.csv
